## Transformer Model for translation English -> Finnish.

In [1]:
import keras
import tensorflow as tf
import numpy as np
from keras import layers
from keras import ops
from keras.saving import register_keras_serializable


In [2]:
text_file = "fin-eng/fin.txt"

with open(text_file, encoding='utf-8') as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    english, finnish, rest = line.split("\t")
    finnish = "[start] " + finnish + " [end]"
    text_pairs.append((english, finnish))

print(text_pairs[:10])

[('Go.', '[start] Mene. [end]'), ('Hi.', '[start] Moro! [end]'), ('Hi.', '[start] Terve. [end]'), ('Run!', '[start] Juokse! [end]'), ('Run!', '[start] Juoskaa! [end]'), ('Run.', '[start] Juokse. [end]'), ('Who?', '[start] Kuka? [end]'), ('Wow!', '[start] Mahtavaa! [end]'), ('Wow!', '[start] Siistiä! [end]'), ('Wow!', '[start] Vau! [end]')]


In [3]:
import random
random.shuffle(text_pairs)
num_val_samples = int(0.15 * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples
train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples:num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples:]

In [4]:
import string
import re

strip_chars = string.punctuation
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")


def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", "")

In [5]:
vocab_size = 15000
sequence_length = 20

source_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)

target_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    # standardize=custom_standardization,
)


train_english_texts = [pair[0] for pair in train_pairs]
train_finnish_texts = [pair[1] for pair in train_pairs]
source_vectorization.adapt(train_english_texts)
target_vectorization.adapt(train_finnish_texts)




In [6]:
batch_size = 64

def format_dataset(eng, fin):
    eng = source_vectorization(eng)
    fin = target_vectorization(fin)
    return ({
        "english": eng,
        "finnish": fin[:, :-1],
    }, fin[:, 1:])

def make_dataset(pairs):
    eng_texts, fin_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    fin_texts = list(fin_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, fin_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset,  num_parallel_calls=4)
    return dataset.shuffle(2048).prefetch(16).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

for inputs, targets in train_ds.take(1):
    print(f"inputs['english'].shape: {inputs['english'].shape}")
    print(f"inputs['finnish'].shape: {inputs['finnish'].shape}")

inputs['english'].shape: (64, 20)
inputs['finnish'].shape: (64, 20)


In [7]:
@register_keras_serializable()
class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.attention_2 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.dense_proj = keras.Sequential([
            layers.Dense(dense_dim, activation="relu"),
            layers.Dense(embed_dim),]
            )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.supports_masking = True

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "dense_dim": self.dense_dim,
        })
        return config

    def get_casual_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, tf.newaxis]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        mask = tf.reshape(mask, (1, input_shape[1], input_shape[1]))  
        mult = tf.concat([tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)], axis=0)
        return tf.tile(mask, mult)
    
    def call(self, inputs, encoder_outputs, mask=None):
        # Create a causal mask for self-attention
        causal_mask = self.get_casual_attention_mask(inputs)

        # Combine with padding mask if provided
        if mask is not None:
            padding_mask = tf.cast(mask[:, tf.newaxis, :], dtype="int32")
            padding_mask = tf.minimum(padding_mask, causal_mask)
        else:
            padding_mask = causal_mask

        # Self-attention (decoder attends to previous tokens)
        attention_output_1 = self.attention_1(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=causal_mask
        )
        attention_output_1 = self.layernorm_1(inputs + attention_output_1)

        # Cross-attention (decoder attends to encoder outputs)
        attention_output_2 = self.attention_2(
            query=attention_output_1,
            value=encoder_outputs,
            key=encoder_outputs,
            attention_mask=padding_mask
        )
        attention_output_2 = self.layernorm_2(attention_output_1 + attention_output_2)

        # Feed-forward network (dense projection)
        proj_output = self.dense_proj(attention_output_2)
        
        # Final layer normalization and return
        return self.layernorm_3(attention_output_2 + proj_output)



In [8]:
@register_keras_serializable()
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(dense_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        if mask is not None:
            padding_mask = ops.cast(mask[:, None, :], dtype="int32")
        else:
            padding_mask = None

        attention_output = self.attention(
            query=inputs, value=inputs, key=inputs, attention_mask=padding_mask
        )
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "dense_dim": self.dense_dim,
                "num_heads": self.num_heads,
            }
        )
        return config

In [9]:
@register_keras_serializable()
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=embed_dim
        )
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def call(self, inputs):
        length = ops.shape(inputs)[-1]
        positions = ops.arange(0, length, 1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return ops.not_equal(inputs, 0)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "sequence_length": self.sequence_length,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config

In [ ]:
embed_dim = 256
dense_dim = 512
num_heads = 8

encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(encoder_inputs)
encoder_outputs = TransformerEncoder(embed_dim, dense_dim, num_heads) (x)

decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="finnish")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(decoder_inputs)
x = TransformerDecoder(embed_dim, dense_dim, num_heads) (x, encoder_outputs)
x = layers.Dropout(0.5) (x)

decoder_outputs = layers.Dense(vocab_size, activation="softmax") (x)
transformer = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

transformer.compile(
 optimizer="adam",
 loss="sparse_categorical_crossentropy",
 metrics=["accuracy"])
transformer.fit(train_ds, epochs=10, validation_data=val_ds)




Epoch 1/10
791/791 ━━━━━━━━━━━━━━━━━━━━ 242s 301ms/step - accuracy: 0.2331 - loss: 4.9400 - val_accuracy: 0.1713 - val_loss: 3.1521
Epoch 2/10
791/791 ━━━━━━━━━━━━━━━━━━━━ 237s 300ms/step - accuracy: 0.1704 - loss: 3.3147 - val_accuracy: 0.1896 - val_loss: 2.5627
Epoch 3/10
791/791 ━━━━━━━━━━━━━━━━━━━━ 237s 300ms/step - accuracy: 0.1891 - loss: 2.6412 - val_accuracy: 0.1990 - val_loss: 2.2859
Epoch 4/10
791/791 ━━━━━━━━━━━━━━━━━━━━ 237s 300ms/step - accuracy: 0.2017 - loss: 2.1793 - val_accuracy: 0.2048 - val_loss: 2.1270
Epoch 5/10
791/791 ━━━━━━━━━━━━━━━━━━━━ 237s 300ms/step - accuracy: 0.2124 - loss: 1.8388 - val_accuracy: 0.2056 - val_loss: 2.0857
Epoch 6/10
791/791 ━━━━━━━━━━━━━━━━━━━━ 237s 299ms/step - accuracy: 0.2218 - loss: 1.5868 - val_accuracy: 0.2085 - val_loss: 2.0514
Epoch 7/10
791/791 ━━━━━━━━━━━━━━━━━━━━ 236s 298ms/step - accuracy: 0.2295 - loss: 1.3937 - val_accuracy: 0.2091 - val_loss: 2.0767
Epoch 8/10
791/791 ━━━━━━━━━━━━━━━━━━━━ 240s 304ms/step - accuracy: 0.2368 

In [11]:
# Load model with explicit custom_objects parameter
loaded_model = keras.saving.load_model(
    "english_finnish_transformer.keras", 
    custom_objects={
        "PositionalEmbedding": PositionalEmbedding,
        "TransformerEncoder": TransformerEncoder,
        "TransformerDecoder": TransformerDecoder
    }
)

c:\Users\patri\miniconda3\envs\myenv\Lib\site-packages\keras\src\layers\layer.py:396: UserWarning: `build()` was called on layer 'positional_embedding_18', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
c:\Users\patri\miniconda3\envs\myenv\Lib\site-packages\keras\src\layers\layer.py:396: UserWarning: `build()` was called on layer 'positional_embedding_19', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
c:\Users\patri\miniconda3\envs\myenv\Lib\site-packages\keras\src\layers\layer.py:396: UserWarning: `bu

In [12]:
import numpy as np
fin_vocab = target_vectorization.get_vocabulary()
fin_index_lookup = dict(zip(range(len(fin_vocab)), fin_vocab))
max_decoded_sentence_length = 20

def decode_sequence(input_sequence):
    tokenized_input_sentence = source_vectorization([input_sequence])
    decoded_sentence = "[start]"
    for i in range(max_decoded_sentence_length):
        tokenized_target_sentence = target_vectorization([decoded_sentence])[:, :-1]
        predictions = transformer([tokenized_input_sentence, tokenized_target_sentence])
        sampled_token_index = np.argmax(predictions[0, i, :])
        sampled_token = fin_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token
        if sampled_token == "[end]":
            break
    return decoded_sentence
    
test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(20):
    input_sentence = random.choice(test_eng_texts)
    print(input_sentence)
    print(decode_sequence(input_sentence))

I thought you'd be more sympathetic.
[start] luulin että olisit lisää end  end end end end          
Did you get that?
[start] saitko tuota end  end end end end end           
The problem was under discussion.
[start] ongelma oli käsiteltävänä end  end end end end           
I love watching you cook.
[start] rakastan sinua ja minä laitan end  end end end          
How can I get in touch with you?
[start] miten voin jäädä sisään [UNK] end  end end end end         
This drawer's stuck.
[start] tämä paikka saa end  end end end end           
We buried the hatchet.
[start] [UNK] sotakirveet end  end end end end end           
Both girls started crying.
[start] kumpikin tytöistä purskahti itkuun end  end end end end end         
Who doesn't like Christmas?
[start] kukapa ei [UNK] end  end end end end           
I'd say the same thing.
[start] sanoisin samoin end  end end end end end end          
Leave my camera alone.
[start] jätä minun isoisäni hyväksyvät end  end end end end          
Yo